# Punto 5 — Introducción a la estadística inferencial

## *"¿La diferencia es real o es azar?"*

**Unidad III — Estadística Descriptiva · Diplomatura en Data Analytics (UNNE)**

> **La deuda que venimos arrastrando.** Cinco veces en esta unidad dijimos *"con
> cinco datos no te podés fiar"*: la media más alta y el CV más bajo de `ISLAND`,
> el 100% del crosstab, el `r` con signo opuesto, y las muestras al azar de n=5.
>
> Siempre fue una sensación, nunca un número. **Este punto la convierte en un
> número.**
>
> Mismo formato de siempre: **qué es → dónde sirve → ⚠️ forzando el concepto**.

## 0 · Setup

In [1]:
import os, sys
sys.path.append("../src")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from estadistica import (intervalo_confianza_formula, intervalo_confianza,
                         comparar_dos_grupos, graficar_bootstrap,
                         comparar_segmentos)

sns.set_theme(style="whitegrid")
pd.set_option("display.width", 110)

df = pd.read_csv("../data/raw/housing.csv")
GRUPO, VALOR = "ocean_proximity", "median_house_value"
print(f"{df.shape[0]:,} filas listas")

20,640 filas listas


## 1 · De la muestra a la población

**Qué es.** En el punto 1 separamos dos verbos:

| | |
|---|---|
| **describir** | *"la media de `NEAR BAY` en estos datos es 259.212"* — un hecho sobre tus filas |
| **inferir** | *"la media de `NEAR BAY` es 259.212"* — una afirmación sobre lo que **no** mediste |

Ahí sólo pudimos **nombrar** la diferencia. Ahora podemos **calcularla**.

**Dónde sirve.** Siempre que quieras decir algo más allá de las filas que tenés
delante — que es, básicamente, siempre.

In [2]:
sub = df[df[GRUPO] == "NEAR BAY"]
print(f"media de NEAR BAY en ESTOS datos: {sub[VALOR].mean():,.0f}")
print(f"n = {len(sub):,}")
print()
print("¿Y la media de TODOS los barrios cerca de la bahía, incluidos los que")
print("no están en el dataset? Eso ya no es un cálculo: es una inferencia.")

media de NEAR BAY en ESTOS datos: 259,212
n = 2,290

¿Y la media de TODOS los barrios cerca de la bahía, incluidos los que
no están en el dataset? Eso ya no es un cálculo: es una inferencia.


## 2 · El intervalo de confianza por la fórmula clásica

**Qué es.** En vez de un número, un rango.

```
                    desvío
   media  ±   z  ·  ──────
                      √n
```

Lo que va después del `±` es el **margen de error**. Con `z = 1,96` el intervalo
es del 95%; con `z = 2,58`, del 99%.

**El error más común del punto:** `desvío / √n` **no es el desvío de los datos**.
Es el desvío de la **media**, y es mucho más chico.

In [3]:
r = intervalo_confianza_formula(sub, VALOR)
print(f"media           = {r['media']:>10,.0f}")
print(f"desvío de los datos = {sub[VALOR].std():>10,.0f}   ← cuánto varían los BARRIOS")
print(f"error estándar  = {r['error_estandar']:>10,.0f}   ← cuánto varía la MEDIA")
print(f"margen de error = {r['margen']:>10,.0f}")
print()
print(f"intervalo 95%   = [{r['inferior']:,.0f} , {r['superior']:,.0f}]")

media           =    259,212
desvío de los datos =    122,819   ← cuánto varían los BARRIOS
error estándar  =      2,567   ← cuánto varía la MEDIA
margen de error =      5,030

intervalo 95%   = [254,182 , 264,243]


#### ⚠️ Forzando el concepto: el margen cae con la RAÍZ de n

Esto es lo que hay que entender de verdad de la fórmula.

In [4]:
filas = []
for n in [25, 50, 100, 250, 500, 1_000, 5_000, 20_640]:
    s = df[VALOR].sample(n, random_state=42) if n < len(df) else df[VALOR]
    m = intervalo_confianza_formula(s.to_frame(name=VALOR), VALOR)
    filas.append({"n": n, "margen": round(m["margen"])})

tabla = pd.DataFrame(filas).set_index("n")
tabla["n × 4"] = tabla.index * 4
tabla["margen si cuadruplicás n"] = (tabla["margen"] / 2).round().astype(int)
print(tabla.to_string())

       margen  n × 4  margen si cuadruplicás n
n                                             
25      51212    100                     25606
50      32778    200                     16389
100     21079    400                     10540
250     13931   1000                      6966
500     10219   2000                      5110
1000     7216   4000                      3608
5000     3185  20000                      1592
20640    1574  82560                       787


> **Para achicar el margen a la mitad no alcanza con el doble de datos: hacen falta
> cuatro veces más.** Es la razón económica por la que las muestras grandes son
> caras, y por la que pasado cierto punto agrandar la muestra deja de convenir.

#### ⚠️ Forzando el concepto: qué NO significa "95% de confianza"

| | |
|---|---|
| ✅ **Significa** | de cada 100 intervalos construidos así, 95 contienen el valor real |
| ❌ **No significa** | que haya 95% de probabilidad de que el valor real esté en **tu** intervalo |

El valor real está o no está. El que varía de muestra en muestra es **el
intervalo**. Es una propiedad del **método**, no de tu resultado particular.

> Es la distinción más resbaladiza del punto y no hace falta más que decirla una
> vez. (Sí existe una escuela —la bayesiana— donde la segunda frase es válida, y
> ahí se llama *intervalo de credibilidad*. No entra en esta unidad.)

## 3 · Bootstrap: la misma idea, sin fórmula

**La idea en una frase:** *tu muestra es tu mejor foto de la población. Si la
re-sorteás con reemplazo miles de veces y calculás el estadístico en cada
re-sorteo, ves cuánto podría variar tu resultado por puro azar.*

El intervalo son los **percentiles** de esas estimaciones. Y todas las piezas ya
las usaron: el `for`, el re-muestreo y `np.percentile` vienen del punto 2.

In [5]:
# Las ocho líneas, a la vista. Es exactamente lo que hace la función.
x = sub[VALOR].dropna().values
n = len(x)
rng = np.random.default_rng(42)

estimaciones = np.empty(2000)
for i in range(2000):
    muestra = rng.choice(x, size=n, replace=True)   # CON reemplazo
    estimaciones[i] = np.mean(muestra)

inferior, superior = np.percentile(estimaciones, [2.5, 97.5])
print(f"intervalo bootstrap 95% = [{inferior:,.0f} , {superior:,.0f}]")

intervalo bootstrap 95% = [254,182 , 264,307]


#### ⚠️ Forzando el concepto: qué pasa si sorteás SIN reemplazo

In [6]:
sin_reemplazo = np.empty(200)
for i in range(200):
    muestra = rng.choice(x, size=n, replace=False)   # ← el error
    sin_reemplazo[i] = np.mean(muestra)

print(f"valores distintos obtenidos: {len(np.unique(sin_reemplazo.round(6)))}")
print(f"todas las estimaciones son iguales: {np.allclose(sin_reemplazo, sin_reemplazo[0])}")
print()
print("Sin reemplazo re-ordenás la misma muestra: obtenés siempre la misma media.")
print("El reemplazo es lo que genera muestras DISTINTAS. No es un detalle.")

valores distintos obtenidos: 1
todas las estimaciones son iguales: True

Sin reemplazo re-ordenás la misma muestra: obtenés siempre la misma media.
El reemplazo es lo que genera muestras DISTINTAS. No es un detalle.


#### ⚠️ Forzando el concepto: ¿y da lo mismo que la fórmula?

In [7]:
f = intervalo_confianza_formula(sub, VALOR)
b = intervalo_confianza(sub, VALOR, n_boot=4000)

print(f"fórmula   [{f['inferior']:>9,.0f} , {f['superior']:>9,.0f}]")
print(f"bootstrap [{b['inferior']:>9,.0f} , {b['superior']:>9,.0f}]")
print()
print(f"difieren en {abs(f['inferior']-b['inferior']):.0f} y "
      f"{abs(f['superior']-b['superior']):.0f} dólares, sobre una media de {f['media']:,.0f}")

fórmula   [  254,182 ,   264,243]
bootstrap [  254,151 ,   264,169]

difieren en 31 y 74 dólares, sobre una media de 259,212


> **Coinciden hasta el dólar, y no es casualidad.** La fórmula supone que la media
> muestral se distribuye normal, y con n = 2.290 eso es cierto.
>
> *"Llegan al mismo lado, pero el bootstrap lo entendés y la fórmula la creés."*
>
> La pregunta que queda picando: **¿y cuando n es chico?** Sección 4.

### 3.1 · Lo que la fórmula no puede hacer

La media tiene una fórmula cerrada para su intervalo. La mediana no. Al bootstrap
le da igual: se cambia una palabra.

In [8]:
for nombre, est in [("media", np.mean), ("mediana", np.median)]:
    r = intervalo_confianza(sub, VALOR, estadistico=est, n_boot=3000)
    print(f"{nombre:>8}: {r['estimacion']:>9,.0f}   [{r['inferior']:>9,.0f} , {r['superior']:>9,.0f}]")

print()
print("Y funcionaría igual con el percentil 90, el CV, o cualquier cosa que",
      "sepas calcular.")

   media:   259,212   [  254,156 ,   264,307]


 mediana:   233,800   [  226,700 ,   241,700]

Y funcionaría igual con el percentil 90, el CV, o cualquier cosa que sepas calcular.


## 4 · La pregunta del curso, por fin contestable

Hasta acá describimos **un** grupo. La pregunta madre de la unidad es sobre **dos**:

> ### ¿El segmento A es realmente distinto del segmento B?

`comparar_dos_grupos()` hace bootstrap de la **diferencia** de medias. La regla de
lectura es una sola:

| | |
|---|---|
| el intervalo **no** cruza el cero | la diferencia se sostiene |
| el intervalo **sí** cruza el cero | no podés afirmar que haya diferencia |

Es la versión conceptual del **contraste de hipótesis** (5c). No hace falta p-valor
ni `H0`/`H1`: la regla del cero hace el mismo trabajo y se entiende sin vocabulario.

In [9]:
for a, b_ in [("NEAR BAY", "NEAR OCEAN"), ("<1H OCEAN", "INLAND"),
              ("ISLAND", "NEAR BAY")]:
    r = comparar_dos_grupos(df, GRUPO, VALOR, a, b_, n_boot=3000)
    print(f"{a:11} vs {b_:11} dif = {r['dif_observada']:>9,.0f}   "
          f"[{r['inferior']:>9,.0f} , {r['superior']:>9,.0f}]")
    print(f"{'':26}{r['conclusion']}")
    print()

NEAR BAY    vs NEAR OCEAN  dif =     9,778   [    2,859 ,    16,438]
                          El intervalo NO incluye 0 → la diferencia entre NEAR BAY y NEAR OCEAN se sostiene.



<1H OCEAN   vs INLAND      dif =   115,279   [  112,495 ,   118,098]
                          El intervalo NO incluye 0 → la diferencia entre <1H OCEAN y INLAND se sostiene.

ISLAND      vs NEAR BAY    dif =   121,228   [   57,813 ,   183,786]
                          El intervalo NO incluye 0 → la diferencia entre ISLAND y NEAR BAY se sostiene.



#### ⚠️ Forzando el concepto (1): "no cruza el cero" ≠ "es importante"

Con muchos datos, una diferencia trivial también deja de cruzar el cero.

In [10]:
# Partimos un grupo grande al azar en dos mitades. Por construcción, no hay
# ninguna diferencia real entre ellas: es el mismo grupo.
grande = df[df[GRUPO] == "<1H OCEAN"].copy()
grande["mitad"] = np.random.default_rng(42).permutation(
    ["A"] * (len(grande) // 2) + ["B"] * (len(grande) - len(grande) // 2))

r = comparar_dos_grupos(grande, "mitad", VALOR, "A", "B", n_boot=3000)
print(f"diferencia entre dos mitades AL AZAR del mismo grupo: {r['dif_observada']:,.0f}")
print(f"intervalo: [{r['inferior']:,.0f} , {r['superior']:,.0f}]")
print(f"¿incluye 0? {r['incluye_cero']}")
print()
print("Acá el intervalo SÍ cruza el cero, que es lo correcto: no hay diferencia.")
print("Pero con n suficientemente grande, hasta una diferencia de 200 dólares")
print("dejaría de cruzarlo. Que sea consistente es una pregunta; que valga la")
print("pena es otra, y la contesta el negocio.")

diferencia entre dos mitades AL AZAR del mismo grupo: -1,780
intervalo: [-6,056 , 2,366]
¿incluye 0? True

Acá el intervalo SÍ cruza el cero, que es lo correcto: no hay diferencia.
Pero con n suficientemente grande, hasta una diferencia de 200 dólares
dejaría de cruzarlo. Que sea consistente es una pregunta; que valga la
pena es otra, y la contesta el negocio.


#### ⚠️ Forzando el concepto (2): el n chico no invalida todo por igual

Acá se paga la deuda de la unidad. `ISLAND` tiene 5 filas.

In [11]:
resumen = []
for g in ["ISLAND", "NEAR BAY", "NEAR OCEAN", "<1H OCEAN", "INLAND"]:
    r = intervalo_confianza(df[df[GRUPO] == g], VALOR, n_boot=3000)
    resumen.append({"grupo": g, "n": r["n"], "media": round(r["estimacion"]),
                    "inferior": round(r["inferior"]), "superior": round(r["superior"]),
                    "ancho": round(r["superior"] - r["inferior"])})

tabla = pd.DataFrame(resumen).set_index("grupo")
print(tabla.to_string())
print()
print(f"El intervalo de ISLAND es {tabla.loc['ISLAND','ancho'] / tabla.loc['<1H OCEAN','ancho']:.0f} veces más ancho que el de <1H OCEAN.")

               n   media  inferior  superior   ancho
grupo                                               
ISLAND         5  380440    319949    442940  122991
NEAR BAY    2290  259212    254156    264307   10152
NEAR OCEAN  2658  249434    244822    253844    9022
<1H OCEAN   9136  240084    237792    242305    4514
INLAND      6551  124805    123116    126469    3353

El intervalo de ISLAND es 27 veces más ancho que el de <1H OCEAN.


**Veintisiete veces más ancho.** Por fin *"no te podés fiar"* tiene unidades.

Pero ahora la pregunta interesante: **¿quiere decir que la media de `ISLAND` no
sirve?**

In [12]:
r = comparar_dos_grupos(df, GRUPO, VALOR, "ISLAND", "NEAR BAY", n_boot=3000)
print(f"ISLAND − NEAR BAY = {r['dif_observada']:,.0f}")
print(f"intervalo: [{r['inferior']:,.0f} , {r['superior']:,.0f}]")
print(f"¿incluye 0? {r['incluye_cero']}")
print()
print("Las cinco islas, completas:")
print("  " + "   ".join(f"{v:,.0f}" for v in sorted(df[df[GRUPO] == "ISLAND"][VALOR])))
print()
print(f"Hasta la más barata está por encima de la media de NEAR BAY ({df[df[GRUPO]=='NEAR BAY'][VALOR].mean():,.0f}).")

ISLAND − NEAR BAY = 121,228
intervalo: [57,813 , 183,786]
¿incluye 0? False

Las cinco islas, completas:
  287,500   300,000   414,700   450,000   450,000

Hasta la más barata está por encima de la media de NEAR BAY (259,212).


> **No.** El intervalo es enorme y **aun así no cruza el cero**: con n=5 se puede
> concluir, porque la diferencia es lo bastante grande como para sobrevivir a la
> incertidumbre.

Ahora la otra cara. Hagamos lo mismo con la **correlación** de ese grupo — el
`r = −0,540` que apareció en la Clase 2.

In [13]:
def boot_corr(sub_df, col_x, col_y, n_boot=4000, random_state=42):
    """Intervalo bootstrap para r. Mismo esquema, otro estadístico."""
    m = sub_df[[col_x, col_y]].dropna().values
    rng = np.random.default_rng(random_state)
    out = []
    for _ in range(n_boot):
        s = m[rng.integers(0, len(m), len(m))]
        if len(np.unique(s[:, 0])) < 2 or len(np.unique(s[:, 1])) < 2:
            continue
        out.append(np.corrcoef(s[:, 0], s[:, 1])[0, 1])
    return np.percentile(out, [2.5, 97.5])

for g in ["ISLAND", "<1H OCEAN"]:
    s = df[df[GRUPO] == g]
    r_obs = s["median_income"].corr(s[VALOR])
    lo, hi = boot_corr(s, "median_income", VALOR)
    print(f"{g:11} n={len(s):>5}  r = {r_obs:+.3f}   intervalo [{lo:+.2f} , {hi:+.2f}]")

ISLAND      n=    5  r = -0.540   intervalo [-1.00 , +0.88]


<1H OCEAN   n= 9136  r = +0.679   intervalo [+0.66 , +0.69]


> ### 📌 El n chico no invalida todo por igual.
>
> **Mismo grupo, mismos cinco datos, dos conclusiones opuestas:**
>
> - su **media** sobrevive: la diferencia contra los otros grupos se sostiene;
> - su **correlación** no: el intervalo cubre casi todo el rango posible de `r`,
>   así que no se puede afirmar ni el signo.
>
> **El intervalo te dice qué conclusión sobrevive y cuál no.** Esa es la respuesta
> madura a la pregunta que arrastramos desde la primera clase.

#### ⚠️ Forzando el concepto (3): y del intervalo mismo, tampoco te fíes tanto

In [14]:
fig, axes = plt.subplots(1, 2, figsize=(13, 3.6))
for ax, g in zip(axes, ["ISLAND", "NEAR BAY"]):
    r = intervalo_confianza(df[df[GRUPO] == g], VALOR, n_boot=4000)
    distintos = len(np.unique(r['distribucion'].round(0)))
    graficar_bootstrap(r, f"{g}  ·  n = {r['n']:,}  ·  {distintos:,} valores distintos", ax=ax)
plt.tight_layout(); plt.show()

/var/folders/pm/3m187rc97hbf14248x1ys6km0000gn/T/ipykernel_1814/1108112673.py:6: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


> Con 5 datos, la media re-muestreada sólo puede caer en **55 lugares**: el
> histograma sale escalonado en vez de una campana. El intervalo se calcula igual,
> pero está construido sobre muy poquitos valores distintos.
>
> **Mirar la distribución es parte del método, no un adorno.**

## 5 · Lo que hay que llevarse del punto 5

1. **Inferir no es adivinar:** es ponerle un número a lo que no sabés.
2. **El margen cae con `√n`**, no con `n`: para achicarlo a la mitad hacen falta cuatro veces más datos.
3. **"95% de confianza"** es una propiedad del método, no de tu intervalo particular.
4. **El bootstrap** reusa lo que ya sabías y funciona con cualquier estadístico.
5. **Si el intervalo cruza el cero**, no podés afirmar la diferencia.
6. **"No cruza el cero" no quiere decir "es importante"**: con n gigante, todo deja de cruzarlo.
7. **El n chico no invalida todo por igual.** El intervalo te dice qué sobrevive.